# CoXAM From the Published Corpus

Runs CoXAM's forward simulation using only what ships in `assets/` — **no AI
model is trained and no explanations are generated**.

`decision_tree_logistic_regression_experiment_workflow.ipynb` is the other half
of the pair: it trains an MLP, fits DT/LR surrogates from that model, then
simulates. This notebook skips both and reads the study's published tables:

| | trained here | source |
|---|---|---|
| AI predictions | no | `assets/ai_dataset/coxam/none.csv` |
| DT / LR explanations | no | `assets/explanations/CoXAM/` |
| feature values | no | `assets/ai_dataset/coxam/values.csv` |
| CoXAM policies | no | `coxam/trained_policies/` checkpoints |

Same `xaikitTest` object and the same `run_experiment` call as the other
notebook — only `source="assets"` and the absence of `train_AI_model` differ.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "tutorials" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import src as xk

from src.virtual_experiment_executor.experiment_simualtion.CoXAM.coxam_trial_executor import (
    COXAM_CORPUS_FEATURES,
    COXAM_CORPUS_FEATURE_ALIASES,
    coxam_available_instance_ids,
)

OUTPUT_DIR = REPO_ROOT / "tutorials" / "coxam_load_default_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

APP_ID = "wine_quality"   # the corpus also ships mushrooms
print("corpus features:", list(COXAM_CORPUS_FEATURES[APP_ID]))
print("loader aliases :", COXAM_CORPUS_FEATURE_ALIASES[APP_ID])

## 2. Create the Workflow and Design

In [ ]:
xaikitTest = xk.xaikitTest(output_dir=OUTPUT_DIR)

XAI_TYPES = ["decision_tree", "logistic_regression"]
TESTED_W_XAI = [True, False]

xaikitTest.add_iv("xai_type", "within", XAI_TYPES, randomization="block")
xaikitTest.add_iv("tested_w_xai", "within", TESTED_W_XAI, randomization="trial")

xaikitTest.add_cv("user_task", ["forward_simulation"])
xaikitTest.add_dv("forward_accuracy", ["continuous"])

xaikitTest.validate_design(show=True)

## 3. Prepare the Dataset on the Corpus's Feature Set

Two details make `source="assets"` usable, and both matter:

- **Names.** The loader spells two features differently from the corpus
  (`chlorides` → `Chlorides`, `Gill Spacing` → `Gill`). Those confirmed
  equivalences live in `COXAM_CORPUS_FEATURE_ALIASES`; anything not listed is
  treated as a genuinely different feature.
- **Order.** `rank_features_by_target=False` is required. The default reorders
  features by target correlation, and the corpus's `a0..a5` are *positional* —
  a permutation would attach every coefficient and tree threshold to the wrong
  feature, so the runner refuses rather than guessing.

In [ ]:
COXAM_WINE_FEATURES = ["Alcohol", "Sulphates", "SO2", "Vinegar Taint", "pH", "chlorides"]

data = xaikitTest.prepare_dataset(
    dataset_id=APP_ID,
    model_type="mlp",
    feature_cols=COXAM_WINE_FEATURES,
    rank_features_by_target=False,   # keep the corpus's positional order
    show_available=False,
    show_summary=True,
)

print("feature order:", list(data.raw_feature_names))

## 4. Generate Trials Over Instances the Corpus Serves

`allowed_instance_ids` keeps every trial to an instance the published tables can
explain — without it a trial could name an instance the corpus never covered.

In [ ]:
instance_ids = coxam_available_instance_ids(APP_ID)
print(f"instances in the corpus: {len(instance_ids)}")

trial_result = xaikitTest.generate_trials(
    model_name="mlp",
    participants_per_between_condition=4,
    allowed_instance_ids=instance_ids,
    counterbalancing_strategy="balanced_latin_square",
    trial_randomization_strategy="balanced",
    output_dir="trials",
    preview_rows=6,
    show=True,
)

print("no AI model trained:", xaikitTest.trained_ai_model is None)

## 5. Simulate With CoXAM

`source="assets"` reads the published surrogates instead of fitting new ones,
which is what lets this run with no trained model. Passing `source="fit"` here
would raise, because fitting needs one.

In [ ]:
xaikitTest.set_cognitive_model(cognitive_model_id="coxam")

simulated = xaikitTest.run_experiment(mode="whole_experiment", source="assets")

print("simulated rows:", len(simulated))
preview_columns = [
    column
    for column in (
        "participantId", "instanceId", "xai_type", "tested_w_xai",
        "selected_strategy", "ai_prediction", "agent_prediction", "forward_accuracy",
    )
    if column in simulated.columns
]
simulated[preview_columns].head(8)

## 6. Results

In [ ]:
print(simulated["selected_strategy"].value_counts().to_string())

simulated.groupby(["xai_type", "tested_w_xai"])["forward_accuracy"].agg(
    ["mean", "size"]
).round(3)

In [ ]:
xaikitTest.plot_results_grid(
    ivs=["xai_type", "tested_w_xai"],
    dvs=["forward_accuracy"],
    phase=None,
    title=f"CoXAM on the published {APP_ID} corpus",
)

## 7. Human vs CoXAM on the Real Study Data

Sections 1-6 simulate a *new* design. This is the study's own fitted results —
the humans beside the model fitted to them — in one call.

**These are two different CoXAM models.** Above, the RL meta-policy picks a
strategy per trial. The published fit is a DDM-with-timing model (`T_enc`,
`ddm_a`, `ddm_s`, `retrieval_threshold`, `lapse`) estimated per participant by
GP optimisation on response NLL and time MAE jointly. The fitted table is what
the paper reports.

In [ ]:
from src.result_visualizer import human_vs_model_report, study_comparison

report_path = human_vs_model_report("coxam", OUTPUT_DIR / "human_vs_coxam.html")
print("wrote", report_path)

`human_vs_model_report()` with no arguments covers every study that has
fitted results on disk. `study_comparison` returns the panels instead of
rendering them, so the numbers can be inspected directly.

In [ ]:
coxam = study_comparison("coxam")
print(coxam.name, "|", coxam.task)
print("participants:", coxam.participants, "| panels:", len(coxam.panels))

for panel in coxam.panels[:2]:
    print()
    print(panel.title)
    print(panel.to_frame().to_string(index=False))

## 8. What This Notebook Shows

- CoXAM runs end to end with **no AI training and no explanation generation** —
  every input is a table or checkpoint already in the repository.
- `coxam_available_instance_ids` is what keeps trials to instances the corpus
  can serve.
- The meta-policy switches strategy per trial: `dt_traversal` under a decision
  tree, `lr_calculation` / `lr_heuristic` under logistic regression.
- `source="assets"` needs the study's features to match the corpus in **both**
  spelling and order — `COXAM_CORPUS_FEATURE_ALIASES` handles the first,
  `rank_features_by_target=False` the second.
- `human_vs_model_report()` renders every study that has fitted results in one
  call; `study_comparison(name)` returns the panels to inspect instead. CoAX is
  replayed through `run_coax_human_replay`, so both its series carry real
  confidence intervals.
- The fitted CoXAM is a different model from the meta-policy simulated above —
  the DDM the paper reports, not the RL agent.

To train your own model and fit surrogates against it instead, use
`decision_tree_logistic_regression_experiment_workflow.ipynb`, which calls the
same `run_experiment` with `source="fit"`.